# 01 — Exploratory Data Analysis
## Care Transition Efficiency & Placement Outcome Analytics

This notebook performs the initial inspection and exploratory data analysis (EDA) of the
`HHS_Unaccompanied_Alien_Children_Program.csv` dataset, prior to any feature engineering or
KPI modeling. It documents data quality findings, distributions, time trends, and
relationships between the five reported flow variables.

**Pipeline stage:** Raw CSV → Validation → Cleaning → **EDA**

See `docs/03_EDA_Findings.md` for the written summary of findings produced from this notebook.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"

from src.data_loader import load_raw_data, inspect_raw_data
from src.preprocessing import run_preprocessing_pipeline
from src.feature_engineering import run_feature_engineering
from src.analysis import descriptive_statistics, correlation_analysis, pairwise_correlation, weekday_comparison, monthly_aggregation
from src.visualizations import (
    line_chart, area_chart, dual_axis_chart, scatter_with_trendline, correlation_heatmap,
    weekday_box_plot, pipeline_flow_chart, rolling_comparison_chart,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

## 1. Initial Inspection of Raw Data

In [2]:
raw_df = load_raw_data()
summary = inspect_raw_data(raw_df)
print("Shape:", summary['shape'])
print("Fully blank rows:", summary['fully_blank_rows'])
print("Duplicate rows (incl. blanks):", summary['duplicate_rows'])
print("\nMissing values per column:")
for k, v in summary['missing_values'].items():
    print(f"  {k}: {v}")
raw_df.head(10)

Shape: (1170, 6)
Fully blank rows: 450
Duplicate rows (incl. blanks): 449

Missing values per column:
  Date: 450
  Children apprehended and placed in CBP custody*: 450
  Children in CBP custody: 450
  Children transferred out of CBP custody: 450
  Children in HHS Care: 450
  Children discharged from HHS Care: 450


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6,18,11,"2,484",14
1,"December 18, 2025",11,50,6,"2,472",16
2,"December 17, 2025",7,31,11,"2,481",10
3,"December 16, 2025",8,54,15,"2,468",9
4,"December 15, 2025",11,42,9,"2,470",7
5,"December 14, 2025",8,35,4,"2,462",8
6,"December 11, 2025",7,47,9,"2,437",10
7,"December 10, 2025",10,54,5,"2,439",9
8,"December 09, 2025",4,30,7,"2,443",8
9,"December 08, 2025",9,27,9,"2,440",4


In [3]:
raw_df.tail(10)

,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
1160,NaN,NaN,NaN,NaN,NaN,NaN
1161,NaN,NaN,NaN,NaN,NaN,NaN
1162,NaN,NaN,NaN,NaN,NaN,NaN
1163,NaN,NaN,NaN,NaN,NaN,NaN
1164,NaN,NaN,NaN,NaN,NaN,NaN
1165,NaN,NaN,NaN,NaN,NaN,NaN
1166,NaN,NaN,NaN,NaN,NaN,NaN
1167,NaN,NaN,NaN,NaN,NaN,NaN
1168,NaN,NaN,NaN,NaN,NaN,NaN
1169,NaN,NaN,NaN,NaN,NaN,NaN


**Finding:** The raw CSV contains 450 fully-blank trailing rows — a CSV export artifact,
not genuine missing observations. These are dropped entirely during preprocessing rather than
treated as missing data requiring imputation.

## 2. Run the Cleaning Pipeline

In [4]:
clean_df, report = run_preprocessing_pipeline(raw_df)
print("Shape after cleaning:", clean_df.shape)
print("\nNumeric validation (negatives / nulls):")
for k, v in report['numeric_validation'].items():
    print(f"  {k}: {v}")
clean_df.head()

Shape after cleaning: (720, 6)

Numeric validation (negatives / nulls):
  apprehensions_negative_count: 0
  apprehensions_null_count: 0
  cbp_custody_negative_count: 0
  cbp_custody_null_count: 0
  transfers_negative_count: 0
  transfers_null_count: 0
  hhs_care_negative_count: 0
  hhs_care_null_count: 0
  discharges_negative_count: 0
  discharges_null_count: 0


,date,apprehensions,cbp_custody,transfers,hhs_care,discharges
0,2023-01-12,33,53,34,6566,436
1,2023-01-22,32,49,39,7122,227
2,2023-01-23,32,50,39,7280,181
3,2023-01-24,47,42,47,7433,175
4,2023-01-25,20,22,41,7538,180


### 2.1 Date Continuity

In [5]:
continuity = report['date_continuity']
print("Date range:", continuity['date_min'].date(), "to", continuity['date_max'].date())
print("Calendar days in range:", continuity['n_calendar_days'])
print("Reported days:", continuity['n_reported_days'])
print("Missing days:", continuity['n_missing_days'])
print("\nReported-day weekday counts:")
for day, count in continuity['weekday_counts_reported'].items():
    print(f"  {day}: {count}")
print("\nMissing-day weekday counts (which weekdays account for the gaps):")
for day, count in continuity['weekday_counts_missing'].items():
    print(f"  {day}: {count}")

Date range: 2023-01-12 to 2025-12-21
Calendar days in range: 1075
Reported days: 720
Missing days: 355

Reported-day weekday counts:
  Tuesday: 149
  Thursday: 147
  Wednesday: 147
  Monday: 145
  Sunday: 130
  Friday: 2

Missing-day weekday counts (which weekdays account for the gaps):
  Saturday: 154
  Friday: 152
  Sunday: 24
  Monday: 8
  Thursday: 7
  Wednesday: 6
  Tuesday: 4


**Finding:** The program's reporting cadence itself skips **Friday and Saturday** almost
entirely (only 2 Fridays and 0 Saturdays are reported across the full dataset). This is a
structural characteristic of how the data is published, not a data-quality defect requiring
correction — no interpolation or forward-filling is applied for these non-reporting days.
The remaining ~49 gaps on otherwise-expected reporting days (Sun–Thu) most likely correspond
to holidays or reporting pauses and are documented as-is.

## 3. Feature Engineering (for EDA convenience)

In [6]:
df = run_feature_engineering(clean_df)
print(df.shape)
df.head()

(720, 35)


,date,apprehensions,cbp_custody,transfers,hhs_care,discharges,Year,Month,Month_Name,Week,Weekday,Day_of_Week,Quarter,transfer_efficiency,discharge_effectiveness,cbp_transfer_rate,net_flow_pressure,cumulative_net_flow_pressure,cbp_transfer_gap,hhs_discharge_gap,transfers_roll7,discharges_roll7,transfer_efficiency_roll7,discharge_effectiveness_roll7,net_flow_pressure_roll7,transfers_roll14,discharges_roll14,transfer_efficiency_roll14,discharge_effectiveness_roll14,net_flow_pressure_roll14,transfers_roll30,discharges_roll30,transfer_efficiency_roll30,discharge_effectiveness_roll30,net_flow_pressure_roll30
0,2023-01-12,33,53,34,6566,436,2023,1,January,2,3,Thursday,1,64.150943,6.640268,103.030303,-403,-403,19,6130,34.000000,436.000000,64.150943,6.640268,-403.00,34.000000,436.000000,64.150943,6.640268,-403.00,34.000000,436.000000,64.150943,6.640268,-403.00
1,2023-01-22,32,49,39,7122,227,2023,1,January,3,6,Sunday,1,79.591837,3.187307,121.875000,-195,-598,10,6895,36.500000,331.500000,71.871390,4.913787,-299.00,36.500000,331.500000,71.871390,4.913787,-299.00,36.500000,331.500000,71.871390,4.913787,-299.00
2,2023-01-23,32,50,39,7280,181,2023,1,January,4,0,Monday,1,78.000000,2.486264,121.875000,-149,-747,11,7099,37.333333,281.333333,73.914260,4.104613,-249.00,37.333333,281.333333,73.914260,4.104613,-249.00,37.333333,281.333333,73.914260,4.104613,-249.00
3,2023-01-24,47,42,47,7433,175,2023,1,January,4,1,Tuesday,1,111.904762,2.354366,100.000000,-128,-875,-5,7258,39.750000,254.750000,83.411886,3.667051,-218.75,39.750000,254.750000,83.411886,3.667051,-218.75,39.750000,254.750000,83.411886,3.667051,-218.75
4,2023-01-25,20,22,41,7538,180,2023,1,January,4,2,Wednesday,1,186.363636,2.387901,205.000000,-160,-1035,-19,7358,40.000000,239.800000,104.002236,3.411221,-207.00,40.000000,239.800000,104.002236,3.411221,-207.00,40.000000,239.800000,104.002236,3.411221,-207.00


## 4. Distributions & Descriptive Statistics

In [7]:
desc = descriptive_statistics(df, columns=['apprehensions','cbp_custody','transfers','hhs_care','discharges'])
desc.round(2)

,count,mean,std,min,25%,50%,75%,max,coefficient_of_variation
apprehensions,720.0,93.52,72.65,0.0,12.00,99.0,147.25,333.0,0.78
cbp_custody,720.0,171.49,126.35,7.0,36.00,193.0,263.25,531.0,0.74
transfers,720.0,128.67,97.32,0.0,14.00,157.0,199.25,440.0,0.76
hhs_care,720.0,6061.28,2833.07,1972.0,2467.75,6406.5,8010.25,11516.0,0.47
discharges,720.0,173.41,125.70,0.0,19.75,181.0,267.00,505.0,0.72


In [8]:
fig = px.histogram(df, x='apprehensions', nbins=40, title='Distribution of Daily Apprehensions')
fig.show()

In [9]:
fig = px.histogram(df, x='hhs_care', nbins=40, title='Distribution of Children in HHS Care (population on given day)')
fig.show()

**Finding:** `hhs_care` represents a population **stock** (children currently in care),
consistently in the thousands, while `discharges` is a daily **flow** typically in the tens
to low hundreds. This scale difference is why `discharge_effectiveness`
(discharges / hhs_care × 100) is structurally a small percentage (median ≈ 2–3%) rather than
a large one — an important consideration for setting realistic bottleneck thresholds later.

## 5. Time Trends — Daily, Weekly, Monthly

In [10]:
fig = pipeline_flow_chart(df)
fig.show()

In [11]:
fig = rolling_comparison_chart(df, 'discharges', 'Discharges: Daily vs Rolling Averages', 'Children discharged')
fig.show()

In [12]:
monthly = monthly_aggregation(df)
fig = line_chart(monthly.rename(columns={'year_month':'date'}).assign(date=lambda d: pd.to_datetime(d['date'])),
                  ['transfer_efficiency','discharge_effectiveness'],
                  'Monthly Average: Transfer Efficiency vs Discharge Effectiveness', 'Percent (%)')
fig.show()

## 6. Relationships Between Variables

In [13]:
corrs = correlation_analysis(df)
print("Pearson correlation matrix:")
display(corrs['pearson'].round(2))
print("\nSpearman correlation matrix:")
display(corrs['spearman'].round(2))

Pearson correlation matrix:


,apprehensions,cbp_custody,transfers,hhs_care,discharges
apprehensions,1.00,0.95,0.89,0.69,0.63
cbp_custody,0.95,1.00,0.93,0.66,0.60
transfers,0.89,0.93,1.00,0.71,0.66
hhs_care,0.69,0.66,0.71,1.00,0.92
discharges,0.63,0.60,0.66,0.92,1.00



Spearman correlation matrix:


,apprehensions,cbp_custody,transfers,hhs_care,discharges
apprehensions,1.00,0.94,0.86,0.66,0.62
cbp_custody,0.94,1.00,0.89,0.63,0.60
transfers,0.86,0.89,1.00,0.66,0.63
hhs_care,0.66,0.63,0.66,1.00,0.89
discharges,0.62,0.60,0.63,0.89,1.00


In [14]:
fig = correlation_heatmap(corrs['pearson'], 'Pearson Correlation — Flow Variables')
fig.show()

In [15]:
pairs = [
    ('cbp_custody', 'transfers', 'CBP Custody vs Transfers'),
    ('hhs_care', 'discharges', 'HHS Care vs Discharges'),
    ('apprehensions', 'transfers', 'Apprehensions vs Transfers'),
]
for a, b, title in pairs:
    stats = pairwise_correlation(df, a, b)
    print(f"{title}: Pearson r={stats['pearson_r']:.3f} (p={stats['pearson_p']:.4f}), "
          f"Spearman r={stats['spearman_r']:.3f} (p={stats['spearman_p']:.4f}), n={stats['n']}")
    fig = scatter_with_trendline(df, a, b, title)
    fig.show()

CBP Custody vs Transfers: Pearson r=0.925 (p=0.0000), Spearman r=0.886 (p=0.0000), n=720


HHS Care vs Discharges: Pearson r=0.921 (p=0.0000), Spearman r=0.895 (p=0.0000), n=720


Apprehensions vs Transfers: Pearson r=0.888 (p=0.0000), Spearman r=0.865 (p=0.0000), n=720


**Note:** Correlation does not imply causation. These relationships describe how the
aggregate daily counts co-move; they do not establish that one stage *causes* movement in
another (see `docs/08_Limitations.md`).

## 7. Weekday Patterns

In [16]:
weekday_stats = weekday_comparison(df)
weekday_stats.round(2)

apprehensions              transfers              discharges              transfer_efficiency              discharge_effectiveness             
                     mean median count      mean median count       mean median count                mean median count                    mean median count
Day_of_Week                                                                                                                                                
Monday              93.78  102.0   145    123.73  155.0   145     157.56  174.0   145               66.05  66.80   145                    2.17   2.60   145
Tuesday             97.11  109.0   149    124.86  146.0   149     136.11  145.0   149               67.05  69.11   149                    1.87   2.21   149
Wednesday           93.80  104.0   147    131.67  160.0   147     166.01  174.0   147               69.20  69.41   147                    2.26   2.66   147
Thursday            94.69  101.0   147    133.41  155.0   147     205.69  213.0   147               73.26  73.68   147                    2.79   3.16   147
Friday              44.50   44.5     2    110.00  110.0     2     144.00  144.0     2               71.18  71.18     2                    2.40   2.40     2
Sunday              88.24   93.5   130    130.06  157.5   130     206.14  231.0   130               70.01  70.99   130                    2.84   3.52   130

In [17]:
fig = weekday_box_plot(df, 'transfer_efficiency', 'Transfer Efficiency by Day of Week')
fig.show()

## 8. Summary of Key EDA Findings

See `docs/03_EDA_Findings.md` for the full written summary. Key points:

1. Raw file contained 450 blank artifact rows; cleaned dataset has 720 valid daily records.
2. Reporting cadence skips Friday/Saturday almost entirely — a structural feature of the data, not a defect.
3. No negative values or impossible percentages were found in the numeric columns.
4. `hhs_care` (population stock) is roughly an order of magnitude (or more) larger than daily flow columns, which materially affects KPI scale (especially Discharge Effectiveness).
5. CBP custody and transfers, and HHS care and discharges, show positive but imperfect correlation — consistent with a multi-stage operational pipeline rather than a single controlled process.